# 🔴 Notebook 1: Redis Data Structures

Redis is a "data structure store" — not just a key-value cache. Each value can be a String, Hash, List, Set, Sorted Set, Stream, or Geospatial Index. Understanding these structures is the foundation for everything else in Redis.

## Learning Objectives
- Understand Redis' core data structures and when to use each
- Practice the most important commands for each data structure
- See real-world use cases: counters, user profiles, task queues, leaderboards, proximity search

## 🛠️ Setup

```bash
cd 03-technologies/databases/redis
docker compose up -d
```

### Visualization
- **RedisInsight**: http://localhost:5540 — connect to `redis://localhost:6379`

### Kernel Selection
Select the `.venv` kernel in VS Code's kernel picker (top-right).
If it doesn't appear, reload: `Cmd+Shift+P` → "Reload Window".

In [1]:
import redis
import json
import time

r = redis.Redis(host="localhost", port=6379, decode_responses=True)

try:
    r.ping()
    print("✅ Connected to Redis")
    # Start fresh
    r.flushdb()
    print("🧹 Flushed database for a clean start")
except Exception as e:
    print(f"❌ Redis connection failed: {e}")
    print("   Run: cd 03-technologies/databases/redis && docker compose up -d")

✅ Connected to Redis
🧹 Flushed database for a clean start


## 1️⃣ Strings
The simplest data type. A key maps to a single value (text, number, or binary data up to 512 MB).

**Real-world uses**: counters (page views, likes), session tokens, feature flags, simple caching.

In [2]:
# SET and GET — the most basic Redis commands
r.set("greeting", "Hello, Redis!")
print("GET greeting:", r.get("greeting"))

# Strings can store numbers too
r.set("page_views", 0)
r.incr("page_views")        # Atomic increment — no race conditions!
r.incr("page_views")
r.incrby("page_views", 10)  # Increment by a specific amount
print("Page views:", r.get("page_views"))  # "12"

# TTL — keys can expire automatically
r.set("session:abc123", "user_42", ex=30)  # expires in 30 seconds
print("Session TTL:", r.ttl("session:abc123"), "seconds")
print("Session value:", r.get("session:abc123"))

# MSET / MGET — set/get multiple keys at once (saves network round-trips)
r.mset({"color:1": "red", "color:2": "blue", "color:3": "green"})
colors = r.mget("color:1", "color:2", "color:3")
print("Colors:", colors)

print("\n💡 INCR is atomic — safe for counters even with thousands of concurrent clients!")

GET greeting: Hello, Redis!
Page views: 12
Session TTL: 30 seconds
Session value: user_42
Colors: ['red', 'blue', 'green']

💡 INCR is atomic — safe for counters even with thousands of concurrent clients!


## 2️⃣ Hashes
A Hash is like a dictionary/object inside a single key. Perfect for storing structured data.

**Real-world uses**: user profiles, product details, session data, configuration.

Think of it this way:
```
Key:   "user:1001"
Value: { "name": "Alice", "email": "alice@example.com", "age": "30" }
```

In [3]:
# HSET — set fields on a hash (like setting properties on an object)
r.hset("user:1001", mapping={
    "name": "Alice",
    "email": "alice@example.com",
    "age": "30",
    "signup_date": "2024-01-15"
})

# HGET — get a single field
print("Name:", r.hget("user:1001", "name"))

# HGETALL — get the entire hash as a dictionary
user = r.hgetall("user:1001")
print("Full user:", user)

# HINCRBY — increment a numeric field atomically
r.hset("user:1001", "login_count", 0)
r.hincrby("user:1001", "login_count", 1)
r.hincrby("user:1001", "login_count", 1)
print("Login count:", r.hget("user:1001", "login_count"))

# HDEL — remove a field
r.hdel("user:1001", "signup_date")
print("After HDEL:", r.hgetall("user:1001"))

print("\n💡 Hashes are memory-efficient for objects — better than storing JSON in a String!")

Name: Alice
Full user: {'name': 'Alice', 'email': 'alice@example.com', 'age': '30', 'signup_date': '2024-01-15'}
Login count: 2
After HDEL: {'name': 'Alice', 'email': 'alice@example.com', 'age': '30', 'login_count': '2'}

💡 Hashes are memory-efficient for objects — better than storing JSON in a String!


## 3️⃣ Lists
Ordered collections (linked lists under the hood). You can push/pop from both ends.

**Real-world uses**: task queues, activity feeds, recent items, message buffers.

```
LPUSH ← [item3, item2, item1] → RPOP
         ↑ head          tail ↑
```

In [4]:
# LPUSH / RPUSH — add items to left or right end
r.rpush("tasks", "send_email", "resize_image", "update_cache")
r.lpush("tasks", "urgent_alert")  # Add to the front (high priority)

# LRANGE — get items by index range (0-based, -1 means last)
all_tasks = r.lrange("tasks", 0, -1)
print("All tasks:", all_tasks)

# LLEN — how many items?
print("Queue length:", r.llen("tasks"))

# LPOP / RPOP — remove and return from left or right
next_task = r.lpop("tasks")
print("Processing:", next_task)
print("Remaining:", r.lrange("tasks", 0, -1))

# Blocking pop — BLPOP waits for an item if the list is empty
# (This is how Redis-based work queues operate)
# r.blpop("tasks", timeout=5)  # Would block up to 5 seconds

# Use case: Keep only the last 5 recent actions
for i in range(10):
    r.lpush("recent:user42", f"action_{i}")
r.ltrim("recent:user42", 0, 4)  # Keep only first 5 (most recent)
print("Recent actions:", r.lrange("recent:user42", 0, -1))

print("\n💡 LPUSH + RPOP = FIFO queue.  LPUSH + LPOP = stack (LIFO).")

All tasks: ['urgent_alert', 'send_email', 'resize_image', 'update_cache']
Queue length: 4
Processing: urgent_alert
Remaining: ['send_email', 'resize_image', 'update_cache']
Recent actions: ['action_9', 'action_8', 'action_7', 'action_6', 'action_5']

💡 LPUSH + RPOP = FIFO queue.  LPUSH + LPOP = stack (LIFO).


## 4️⃣ Sets
Unordered collections of unique strings. Great for membership tests and set operations.

**Real-world uses**: tags, unique visitors, friend lists, "already seen" deduplication.

In [5]:
# SADD — add members to a set
r.sadd("user:1001:interests", "python", "redis", "hiking", "cooking")
r.sadd("user:1002:interests", "redis", "cooking", "gaming", "music")

# SMEMBERS — get all members
print("Alice's interests:", r.smembers("user:1001:interests"))

# SISMEMBER — check if an item exists (O(1) — very fast!)
print("Alice likes redis?", r.sismember("user:1001:interests", "redis"))
print("Alice likes gaming?", r.sismember("user:1001:interests", "gaming"))

# SCARD — count of unique members
print("Alice's interest count:", r.scard("user:1001:interests"))

# Set operations — these are incredibly powerful
common = r.sinter("user:1001:interests", "user:1002:interests")
print("Common interests:", common)

only_alice = r.sdiff("user:1001:interests", "user:1002:interests")
print("Only Alice:", only_alice)

all_interests = r.sunion("user:1001:interests", "user:1002:interests")
print("All interests:", all_interests)

print("\n💡 SINTER is perfect for 'mutual friends' or 'shared tags' features!")

Alice's interests: {'cooking', 'redis', 'python', 'hiking'}
Alice likes redis? 1
Alice likes gaming? 0
Alice's interest count: 4
Common interests: {'redis', 'cooking'}
Only Alice: {'python', 'hiking'}
All interests: {'redis', 'python', 'music', 'hiking', 'gaming', 'cooking'}

💡 SINTER is perfect for 'mutual friends' or 'shared tags' features!


## 5️⃣ Sorted Sets (ZSets)
Like Sets, but every member has a **score**. Members are kept in score order. This makes Sorted Sets perfect for rankings and leaderboards.

**Real-world uses**: leaderboards, priority queues, time-series indexes, "top N" queries.

In [6]:
# ZADD — add members with scores
# Imagine a gaming leaderboard
r.zadd("leaderboard:game1", {
    "Alice": 2500,
    "Bob": 1800,
    "Charlie": 3200,
    "Diana": 2900,
    "Eve": 2100,
})

# ZRANGE — get members sorted by score (ascending)
print("Leaderboard (low to high):")
for rank, (player, score) in enumerate(
    r.zrange("leaderboard:game1", 0, -1, withscores=True), 1
):
    print(f"  #{rank} {player}: {int(score)} pts")

# ZREVRANGE — get members sorted by score (descending) — top players first
print("\n🏆 Top 3 Players:")
for rank, (player, score) in enumerate(
    r.zrevrange("leaderboard:game1", 0, 2, withscores=True), 1
):
    medal = ["🥇", "🥈", "🥉"][rank - 1]
    print(f"  {medal} #{rank} {player}: {int(score)} pts")

# ZRANK / ZREVRANK — get a player's rank
print(f"\nAlice's rank: #{r.zrevrank('leaderboard:game1', 'Alice') + 1}")

# ZINCRBY — update a score atomically
r.zincrby("leaderboard:game1", 500, "Bob")  # Bob got a bonus!
print(f"Bob's new score: {int(r.zscore('leaderboard:game1', 'Bob'))}")

print("\n💡 All operations are O(log N) — fast even with millions of players!")

Leaderboard (low to high):
  #1 Bob: 1800 pts
  #2 Eve: 2100 pts
  #3 Alice: 2500 pts
  #4 Diana: 2900 pts
  #5 Charlie: 3200 pts

🏆 Top 3 Players:
  🥇 #1 Charlie: 3200 pts
  🥈 #2 Diana: 2900 pts
  🥉 #3 Alice: 2500 pts

Alice's rank: #3
Bob's new score: 2300

💡 All operations are O(log N) — fast even with millions of players!


In [7]:
# Real-world pattern: keep only top N entries to save memory
print("Before trim:", r.zcard("leaderboard:game1"), "players")
r.zremrangebyrank("leaderboard:game1", 0, -4)  # Keep only top 3
print("After trim:", r.zcard("leaderboard:game1"), "players")
print("Remaining:", r.zrevrange("leaderboard:game1", 0, -1, withscores=True))

print("\n💡 ZREMRANGEBYRANK is useful for keeping capped leaderboards or top-N lists.")

Before trim: 5 players
After trim: 3 players
Remaining: [('Charlie', 3200.0), ('Diana', 2900.0), ('Alice', 2500.0)]

💡 ZREMRANGEBYRANK is useful for keeping capped leaderboards or top-N lists.


## 6️⃣ Streams
Append-only logs (similar to Kafka topics). Each entry has an auto-generated ID and a set of field-value pairs. Streams are the foundation for durable messaging in Redis.

**Real-world uses**: event sourcing, audit logs, activity feeds, work queues with acknowledgment.

We'll cover Streams in depth in **Notebook 2** — here's a quick taste.

In [8]:
# XADD — append an event to a stream
# The "*" tells Redis to auto-generate a timestamp-based ID
r.xadd("events:orders", {"action": "created", "order_id": "1001", "amount": "59.99"})
r.xadd("events:orders", {"action": "paid", "order_id": "1001", "method": "credit_card"})
r.xadd("events:orders", {"action": "created", "order_id": "1002", "amount": "129.00"})
r.xadd("events:orders", {"action": "shipped", "order_id": "1001", "carrier": "FedEx"})

# XLEN — how many entries?
print("Stream length:", r.xlen("events:orders"))

# XRANGE — read entries (oldest to newest)
print("\nOrder events:")
for entry_id, fields in r.xrange("events:orders"):
    print(f"  [{entry_id}] {fields}")

# XREVRANGE — read newest first
print("\nLatest event:")
latest = r.xrevrange("events:orders", count=1)
print(f"  {latest[0]}")

print("\n💡 Streams are durable — data persists even if consumers disconnect!")
print("   We'll explore consumer groups and work queues in Notebook 2.")

Stream length: 4

Order events:
  [1776555979406-0] {'action': 'created', 'order_id': '1001', 'amount': '59.99'}
  [1776555979406-1] {'action': 'paid', 'order_id': '1001', 'method': 'credit_card'}
  [1776555979407-0] {'action': 'created', 'order_id': '1002', 'amount': '129.00'}
  [1776555979408-0] {'action': 'shipped', 'order_id': '1001', 'carrier': 'FedEx'}

Latest event:
  ('1776555979408-0', {'action': 'shipped', 'order_id': '1001', 'carrier': 'FedEx'})

💡 Streams are durable — data persists even if consumers disconnect!
   We'll explore consumer groups and work queues in Notebook 2.


## 7️⃣ Geospatial Indexes (Bonus)
Redis can store locations and find nearby items. Under the hood, it uses Sorted Sets with geohash encoding.

**Real-world uses**: "find restaurants near me", driver matching (Uber), store locators.

In [9]:
# GEOADD — add locations (longitude, latitude, member)
# GEOADD takes a flat list: lon, lat, name, lon, lat, name, ...
r.geoadd("restaurants", [
    -73.985428, 40.748817, "Empire Diner",        # NYC
    -73.968285, 40.785091, "Central Park Cafe",
    -73.993896, 40.750580, "Hudson Yards Grill",
    -74.006015, 40.714270, "Brooklyn Bridge Deli",
    -73.935242, 40.730610, "Queens Kitchen",
])

# GEODIST — distance between two locations
dist = r.geodist("restaurants", "Empire Diner", "Central Park Cafe", unit="km")
print(f"Distance: Empire Diner ↔ Central Park Cafe = {dist:.2f} km")

# GEOSEARCH — find locations within a radius
nearby = r.geosearch(
    "restaurants",
    longitude=-73.985,
    latitude=40.749,
    radius=3,
    unit="km",
    withcoord=True,
    withdist=True,
    sort="ASC"  # closest first
)

print("\n📍 Restaurants within 3 km of Times Square:")
for item in nearby:
    name = item[0]
    dist_km = item[1]
    coords = item[2]
    print(f"  {name}: {dist_km:.2f} km away (lat={coords[1]:.4f}, lon={coords[0]:.4f})")

print("\n💡 GEOSEARCH is O(N+log(M)) — fast enough for real-time proximity queries!")

Distance: Empire Diner ↔ Central Park Cafe = 4.29 km

📍 Restaurants within 3 km of Times Square:
  Empire Diner: 0.04 km away (lat=40.7488, lon=-73.9854)
  Hudson Yards Grill: 0.77 km away (lat=40.7506, lon=-73.9939)

💡 GEOSEARCH is O(N+log(M)) — fast enough for real-time proximity queries!


## 📋 Data Structure Cheat Sheet

| Structure | Key Commands | Time Complexity | Best For |
|-----------|-------------|-----------------|----------|
| **String** | SET, GET, INCR, EXPIRE | O(1) | Counters, flags, simple cache |
| **Hash** | HSET, HGET, HGETALL, HINCRBY | O(1) per field | Objects, profiles, config |
| **List** | LPUSH, RPOP, LRANGE, LTRIM | O(1) push/pop, O(N) range | Queues, feeds, recent items |
| **Set** | SADD, SISMEMBER, SINTER | O(1) add/check, O(N) ops | Tags, unique visitors, dedup |
| **Sorted Set** | ZADD, ZRANGE, ZREVRANK | O(log N) | Leaderboards, rankings |
| **Stream** | XADD, XREAD, XRANGE | O(1) add, O(N) read | Event logs, message queues |
| **Geo** | GEOADD, GEOSEARCH | O(N+log M) search | Proximity, location features |

## 🧹 Cleanup

In [10]:
r.flushdb()
print("🧹 Cleaned up all keys from this notebook")

🧹 Cleaned up all keys from this notebook


## 📚 Summary

### Key Takeaways

1. **Redis is a data structure store** — not just key-value. Choosing the right structure is key.
2. **Strings** are the simplest — use INCR for atomic counters
3. **Hashes** store objects efficiently — better than JSON strings for partial updates
4. **Lists** make great queues — LPUSH + RPOP for FIFO, BLPOP for blocking
5. **Sets** enable powerful membership tests and operations — SINTER for "mutual friends"
6. **Sorted Sets** are Redis' superpower — O(log N) leaderboards that scale
7. **Streams** provide durable, append-only logs — Redis' answer to Kafka
8. **Geo indexes** enable proximity search — perfect for location-based features

### Next Up

In **Notebook 2**, we'll explore **Pub/Sub and Streams** — Redis' messaging patterns for real-time communication and durable event processing.